In [ ]:
import json
import numpy as np
import tensorflow as tf
import pickle
from sklearn.preprocessing import LabelEncoder

# ======================
# LOAD DATA
# ======================

with open("/content/Dataset.json") as f:
    data = json.load(f)

sentences = []
labels = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        sentences.append(pattern.lower())
        labels.append(intent["tag"])

# ======================
# TOKENIZER
# ======================

tokenizer = tf.keras.preprocessing.text.Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
max_len = max(len(x) for x in sequences)

padded = tf.keras.preprocessing.sequence.pad_sequences(
    sequences, maxlen=max_len, padding="post"
)

# ======================
# LABEL ENCODER
# ======================

encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

# ======================
# MODEL
# ======================

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(len(tokenizer.word_index)+1, 128, input_length=max_len),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(set(labels)), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

model.fit(padded, labels_encoded, epochs=200)

# ======================
# SAVE EVERYTHING
# ======================

model.save("model.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Training complete. Model saved.")




import json
import numpy as np
import tensorflow as tf
import pickle
import random

# Load model and tools
model = tf.keras.models.load_model("model.keras")

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

with open("/content/Dataset.json") as f:
    data = json.load(f)

max_len = model.input_shape[1]

# Predict function
def predict_intent(text):
    sequence = tokenizer.texts_to_sequences([text.lower()])
    padded = tf.keras.preprocessing.sequence.pad_sequences(
        sequence, maxlen=max_len, padding="post"
    )

    prediction = model.predict(padded, verbose=0)[0]
    confidence = np.max(prediction)

    ERROR_THRESHOLD = 0.75

    if confidence > ERROR_THRESHOLD:
        tag = encoder.inverse_transform([np.argmax(prediction)])[0]
        return tag
    else:
        return "unknown"

# Get response
def get_response(tag):
    for intent in data["intents"]:
        if intent["tag"] == tag:
            return random.choice(intent["responses"])
    return "I'm not sure I understood that. Please ask about a specific room or block."

# Chat loop
print("LICET Smart Navigation Bot Ready.")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break

    tag = predict_intent(user_input)
    response = get_response(tag)

    print("Bot:", response)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.0663 - loss: 2.6408
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.1430 - loss: 2.6362
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2197 - loss: 2.6317 
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.1941 - loss: 2.6262
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.1837 - loss: 2.6217
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.2140 - loss: 2.6155
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2197 - loss: 2.6102
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2301 - loss: 2.6063
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3011 - loss: 2.5968
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2140 - loss: 2.5917
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2964 - loss: 2.5802
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.2803 - l